# Inference

Loads the registered pipeline from MLflow and runs it on the raw test set. The pipeline does its own preprocessing, so we feed in raw DataFrame rows.

## 1. Setup

In [1]:
import sys, os, warnings, logging, subprocess
warnings.filterwarnings('ignore')
logging.getLogger('mlflow').setLevel(logging.ERROR)

if os.path.isdir('/kaggle/working'):
    subprocess.run(['pip', 'install', '-q', 'dagshub', 'mlflow', 'xgboost'], check=True)
    REPO = '/kaggle/working/ML_Asgn2'
    if os.path.isdir(REPO):
        subprocess.run(['git', '-C', REPO, 'pull', '--quiet'], check=True)
    else:
        subprocess.run(['git', 'clone', 'https://github.com/Saba0033/ML_Asgn2.git', REPO], check=True)
    os.chdir(REPO)
    from kaggle_secrets import UserSecretsClient
    os.environ['DAGSHUB_USER_TOKEN'] = UserSecretsClient().get_secret('DAGSHUB_TOKEN')

for p in ['.', '..', '/kaggle/working/ML_Asgn2', '/kaggle/working']:
    if os.path.isdir(os.path.join(p, 'src')) and p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from src.data_utils import load_test
from src.mlflow_utils import init_tracking

init_tracking('Inference')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.1 requires dacite<2,>=1.9, but you have dacite 1.6.0 which is incompatible.
Cloning into '/kaggle/working/ML_Asgn2'...


Accessing as Saba0033

Initialized MLflow to track repo "Saba0033/ML_Asgn2"

Repository Saba0033/ML_Asgn2 initialized!

2026/05/06 19:36:45 INFO mlflow.tracking.fluent: Experiment with name 'Inference' does not exist. Creating a new experiment.


  MLflow experiment: Inference
  Tracking URI:      https://dagshub.com/Saba0033/ML_Asgn2.mlflow


## 2. Load registered pipeline

In [2]:
MODEL_NAME = 'IEEEFraudBestModel'
MODEL_VERSION = 'latest'   # or pin to a specific integer version like '3'

model_uri = f'models:/{MODEL_NAME}/{MODEL_VERSION}'
pipeline = mlflow.sklearn.load_model(model_uri)
print(f'Loaded {model_uri}')
print(f'Pipeline steps: {[name for name, _ in pipeline.steps]}')

Loaded models:/IEEEFraudBestModel/latest
Pipeline steps: ['eng', 'pre', 'sel', 'clf']


## 3. Load raw test set

In [3]:
X_test, transaction_ids = load_test()
print(f'test rows: {len(X_test):,}, columns: {X_test.shape[1]}')

  memory: 2164.1 MB -> 1386.1 MB (35.9% reduction)
test rows: 506,691, columns: 432


## 4. Predict and build submission

In [4]:
# Pipeline preprocesses internally, so we feed it the raw frame directly.
fraud_proba = pipeline.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'TransactionID': transaction_ids,
    'isFraud': fraud_proba,
})
os.makedirs('submissions', exist_ok=True)
submission_path = 'submissions/submission.csv'
submission.to_csv(submission_path, index=False)

print(f'wrote {submission_path}  ({len(submission):,} rows)')
print(submission.head())

wrote submissions/submission.csv  (506,691 rows)
   TransactionID   isFraud
0        3663549  0.001740
1        3663550  0.002303
2        3663551  0.003589
3        3663552  0.003931
4        3663553  0.001815


## 5. Architecture comparison

Each `model_experiment_*.ipynb` writes its headline numbers to `results_cache.json` and registers itself as `IEEEFraudBestModel` only if its CV ROC-AUC beats the current registry champion. The cell below prints the leaderboard and confirms which run is currently in the registry.

In [5]:
import json
from src.mlflow_utils import load_architecture_results, REGISTERED_MODEL

cache = load_architecture_results()
if not cache:
    print('results_cache.json is empty; run the model_experiment_*.ipynb notebooks first.')
else:
    rows = []
    for arch, d in cache.items():
        rows.append({
            'architecture':   arch,
            'cv_val_roc_auc': round(d.get('cv_val_roc_auc_mean', float('nan')), 4),
            'cv_val_pr_auc':  round(d.get('cv_val_pr_auc_mean',  float('nan')), 4),
            'cv_std':         round(d.get('cv_val_roc_auc_std',  float('nan')), 4),
            'overfit_gap':    round(d.get('overfit_gap',         float('nan')), 4),
            'best_selector':  d.get('best_selector', ''),
            'n_features':     d.get('n_features_kept', ''),
        })
    leaderboard = (pd.DataFrame(rows)
                   .sort_values('cv_val_roc_auc', ascending=False)
                   .reset_index(drop=True))
    print('Leaderboard (sorted by CV ROC-AUC):')
    print(leaderboard.to_string(index=False))

client = mlflow.MlflowClient()
try:
    versions = client.search_model_versions(f"name='{REGISTERED_MODEL}'")
except Exception:
    versions = []
if versions:
    latest = max(versions, key=lambda v: int(v.version))
    run = client.get_run(latest.run_id)
    print(f"\nRegistry champion: {REGISTERED_MODEL} v{latest.version}")
    print(f"  run_id:        {latest.run_id}")
    print(f"  run_name:      {run.data.tags.get('mlflow.runName', '')}")
    print(f"  cv_val_roc_auc_mean: {run.data.metrics.get('cv_val_roc_auc_mean', float('nan')):.4f}")
else:
    print(f"\nRegistry is empty — no '{REGISTERED_MODEL}' version found yet.")

results_cache.json is empty; run the model_experiment_*.ipynb notebooks first.

Registry champion: IEEEFraudBestModel v3
  run_id:        f06222e1938b4a6982718c311ba72b5a
  run_name:      XGBoost_FinalPipeline
  cv_val_roc_auc_mean: 0.9354
